In [ ]:
from pathlib import Path
import json
import os

from dotenv import load_dotenv
from openpyxl import load_workbook

from agent import ClassifiedSource, classify_source

load_dotenv()

INPUT_FILE = Path("data/sources.xlsx")
SHEET_NAME = None  # None — активный лист
SKIP_FILLED = True

if not os.getenv("OPENAI_MODEL"):
    raise RuntimeError("Добавьте OPENAI_MODEL в .env")

In [ ]:
workbook = load_workbook(INPUT_FILE)
sheet = workbook[SHEET_NAME] if SHEET_NAME else workbook.active
headers = {
    str(cell.value).strip(): cell.column
    for cell in sheet[1]
    if cell.value is not None
}

if "url" not in headers:
    raise ValueError("В XLSX нет колонки 'url'")

url_column = headers["url"]
labels_column = headers.get("labels", sheet.max_column + 1)
sheet.cell(1, labels_column).value = "labels"

In [ ]:
for row in range(2, sheet.max_row + 1):
    url = sheet.cell(row, url_column).value
    labels_cell = sheet.cell(row, labels_column)
    if not url or (SKIP_FILLED and labels_cell.value):
        continue

    url = str(url).strip()
    try:
        result = await classify_source(url)
    except Exception as exc:
        result = ClassifiedSource(
            input_url=url,
            explanation=f"Ошибка обработки: {exc}",
            classified=False,
        )

    labels_cell.value = json.dumps(result.model_dump(), ensure_ascii=False)
    workbook.save(INPUT_FILE)
    print(f"{row - 1}/{sheet.max_row - 1}: {url} — {result.classified}")